# Selah — Scripture at the right physiological moment
### The engine, end-to-end · YouVersion Platform API + Gloo AI Studio API

Every second of a workout, Selah turns a biometric stream into a Scripture delivery:

> **sense** the physiological moment → **discern** the verse built for it (YouVersion) →
> **deliver** one personal line in the moment's native format (Gloo AI).

This notebook runs the whole pipeline against the competition's `biometric movements.csv`
— **5 sessions, 4 activities (running · cycling · HIIT · weightlifting), 14 moment types.**

**Demo mode is ON by default — no keys needed.** It serves the *public-domain* **World
English Bible (WEB)** as an embedded fallback so the notebook runs anywhere and stays
IP-safe. Add your hackathon keys and set `DEMO_MODE = False` to run against the live APIs.

> **Production integration.** The public web demo calls a deployed Cloudflare Worker
> proxy — **`selah-proxy.petitgen.workers.dev`** — which holds both keys server-side, does
> the Gloo OAuth2 token exchange, and serves the **Berean Standard Bible (BSB, bibleId
> 3034 — public domain)** live via YouVersion. `proxy/src/worker.js` is the reference
> implementation; the live branches below mirror it. Gloo access is currently gated by a
> payment-processor decline, so the proxy runs an **honest, clearly-labeled simulation**
> (`source: "gloo-sim"`) that flips to the real API the instant credentials are set — see
> `docs/GLOO-STATUS.md`.

In [ ]:
# ═══ 1 · Config ═══════════════════════════════════════════════════════
import os, json
YOUVERSION_API_KEY = os.environ.get("YOUVERSION_API_KEY", "")   # 🔑 YouVersion Platform App Key
GLOO_AI_API_KEY    = os.environ.get("GLOO_AI_API_KEY", "")      # 🔑 Gloo bearer token, OR "CLIENT_ID:CLIENT_SECRET"
DEMO_MODE          = True   # 🔑 set False once both keys are set (env vars or here)

# Live endpoints — mirror proxy/src/worker.js (the deployed reference integration).
YOUVERSION_API_BASE = "https://api.youversion.com/v1"                    # auth header: X-YVP-App-Key (NOT Bearer)
YV_BIBLE_ID         = 3034                                               # English default: Berean Standard Bible (BSB, public domain)
GLOO_TOKEN_URL      = "https://platform.ai.gloo.com/oauth2/token"        # OAuth2 client-credentials → bearer token
GLOO_AI_API_BASE    = "https://platform.ai.gloo.com/ai/v2"              # → /chat/completions (OpenAI-compatible, auto-routed)
SELAH_PROXY         = "https://selah-proxy.petitgen.workers.dev"        # production: holds keys server-side, does the token exchange

try:
    import requests
except ImportError:
    requests = None
print("Selah · DEMO MODE (public-domain Scripture)" if DEMO_MODE
      else "Selah · LIVE MODE (YouVersion + Gloo)")

## The engine — one source of truth

The cell below is `engine/selah_engine.py` verbatim: the sense→discern→deliver core
that the notebook, the test-suite, and the web demo all share. It is dependency-free.

Key contracts it exposes:
- `classify_moment(...)` — the transparent, activity-aware on-device classifier a watch runs.
- `get_verse(ref, ..., live=…)` — **demo** returns public-domain (WEB) text; **live** calls YouVersion.
- `personalize(moment, verse, ..., live=…)` — **demo** pastoral template; **live** calls Gloo AI.
- `deliver(snapshot, ..., live=…)` — the full loop, once per biometric snapshot.

In [ ]:
# ═══ 2 · engine/selah_engine.py — inlined so the notebook is self-contained ═══
"""Selah engine — the single source of truth for the pipeline.

sense (physiological moment) → discern (verse, via YouVersion) → deliver (Gloo, native format).

Dependency-free and testable. The notebook imports this; the test-suite tests it;
the web demo mirrors its data. Scripture text for offline demo mode is **public
domain** (World English Bible) to stay IP-safe (Rules §3.14); live mode serves the
reader's chosen translation via the YouVersion Platform API.

The deployed Cloudflare Worker proxy/src/worker.js (selah-proxy.petitgen.workers.dev)
is the REFERENCE live integration — it holds the keys server-side, does the Gloo OAuth2
token exchange, and is what the web demo calls. The live branches below mirror that flow.
"""
from __future__ import annotations
from dataclasses import dataclass

# ── moment → verse + theme (extends the hackathon's verse movement mapping) ──
MOMENT_VERSE = {
    "warmup":            ("PSA.118.24", "gratitude"),
    "pre_workout":       ("PSA.118.24", "gratitude"),
    "early_push":        ("JOS.1.9",    "courage"),
    "steady_state":      ("PSA.23.4",   "presence"),
    "working_set":       ("PRO.3.5",    "trust"),
    "breakthrough_wall": ("PHI.4.13",   "strength"),
    "peak_effort":       ("ISA.40.31",  "endurance"),
    "redline":           ("ROM.8.37",   "victory"),
    "final_rep":         ("2CO.12.9",   "grace"),
    "finishing_strong":  ("GAL.6.9",    "perseverance"),
    "rest_set":          ("ISA.41.10",  "renewal"),
    "recovery_window":   ("PSA.46.10",  "peace"),
    "active_recovery":   ("LAM.3.22",   "renewal"),
    "post_workout":      ("1CO.9.24",   "purpose"),
}

# ── how a watch should speak at each moment ──
# mode: "interrupt" = haptic + a verse the runner reads now;
#       "ambient"   = never interrupts — the watch just breathes a color / glow.
DELIVERY = {
    "warmup":            ("gentle display",        "ambient"),
    "pre_workout":       ("gentle display",        "ambient"),
    "early_push":        ("display",               "ambient"),
    "steady_state":      ("ambient display",       "ambient"),
    "working_set":       ("ambient display",       "ambient"),
    "breakthrough_wall": ("haptic pulse + audio",  "interrupt"),
    "peak_effort":       ("haptic pulse + display","interrupt"),
    "redline":           ("haptic pulse + display","interrupt"),
    "final_rep":         ("haptic pulse + display","interrupt"),
    "finishing_strong":  ("haptic pulse + display","interrupt"),
    "rest_set":          ("ambient glow",          "ambient"),
    "recovery_window":   ("ambient glow",          "ambient"),
    "active_recovery":   ("ambient glow",          "ambient"),
    "post_workout":      ("ambient glow",          "ambient"),
}

# ── theme → glow (physiological colour the interface becomes) ──
THEME_COLOR = {
    "gratitude":"#f0a6c0,#e0729a", "courage":"#ff9d5c,#ff6a2a", "presence":"#38e1c0,#14b8a6",
    "trust":"#a98bff,#7c5cff", "strength":"#6f8cff,#3d5bff", "endurance":"#ffb347,#ff7a1a",
    "victory":"#ffd34d,#f5a623", "grace":"#7fd8ff,#38b6ff", "perseverance":"#ffd34d,#f5a623",
    "renewal":"#8ee6b0,#37c98a", "peace":"#8ee6b0,#37c98a", "purpose":"#ffd34d,#f5a623",
}

# ── public-domain verse text for offline DEMO mode (World English Bible) ──
# Live mode serves the reader's own translation via YouVersion (licensed).
DEMO_VERSES_PD = {
 "PSA.118.24":"This is the day that Yahweh has made. We will rejoice and be glad in it!",
 "JOS.1.9":"Haven't I commanded you? Be strong and courageous. Don't be afraid, neither be dismayed, for Yahweh your God is with you wherever you go.",
 "PSA.23.4":"Even though I walk through the valley of the shadow of death, I will fear no evil, for you are with me.",
 "PRO.3.5":"Trust in Yahweh with all your heart, and don't lean on your own understanding.",
 "PHI.4.13":"I can do all things through Christ, who strengthens me.",
 "ISA.40.31":"But those who wait for Yahweh will renew their strength. They will mount up with wings like eagles. They will run, and not be weary.",
 "ROM.8.37":"No, in all these things, we are more than conquerors through him who loved us.",
 "2CO.12.9":"My grace is sufficient for you, for my power is made perfect in weakness.",
 "GAL.6.9":"Let us not be weary in doing good, for we will reap in due season, if we don't give up.",
 "ISA.41.10":"Don't be afraid, for I am with you. Don't be dismayed, for I am your God. I will strengthen you.",
 "PSA.46.10":"Be still, and know that I am God.",
 "LAM.3.22":"It is because of Yahweh's loving kindnesses that we are not consumed, because his compassion doesn't fail.",
 "1CO.9.24":"Don't you know that those who run in a race all run, but one receives the prize? Run like that, so that you may win.",
}
VERSE_NAME = {
 "PSA.118.24":"Psalm 118:24","JOS.1.9":"Joshua 1:9","PSA.23.4":"Psalm 23:4","PRO.3.5":"Proverbs 3:5",
 "PHI.4.13":"Philippians 4:13","ISA.40.31":"Isaiah 40:31","ROM.8.37":"Romans 8:37","2CO.12.9":"2 Corinthians 12:9",
 "GAL.6.9":"Galatians 6:9","ISA.41.10":"Isaiah 41:10","PSA.46.10":"Psalm 46:10","LAM.3.22":"Lamentations 3:22",
 "1CO.9.24":"1 Corinthians 9:24",
}

# ── faith-tuned one-liners for demo mode (Gloo shapes these live) ──
DEMO_NOTES = {
 "breakthrough_wall":"This is the wall, Maya. You were built to go through it.",
 "peak_effort":"Right here, at the top — new strength. Keep running.",
 "redline":"All the way through the red. More than a conqueror.",
 "final_rep":"One more. His power shows up right where you're weakest.",
 "finishing_strong":"The last mile is the offering. Don't give up now.",
 "recovery_window":"Be still, Maya. You did enough.",
 "post_workout":"You ran your race today. Well done.",
}


@dataclass
class Snapshot:
    heart_rate: int; hr_zone: int; effort_pct: float
    recovery_score: int = 70; stress_index: float = 2.0
    activity_type: str = "running"; session_minute: int = 0
    translation: str = "WEB"; language: str = "en"


def classify_moment(hr_zone: int, effort_pct: float, activity: str = "running",
                    minute: int = 5, session_len: int = 30) -> str:
    """Transparent, on-device moment classifier (activity-aware).

    The notebook additionally trains + validates a RandomForest on the provided
    sessions; this rule-anchored version is the tiny model a watch would run."""
    if minute == 0:                          return "warmup"
    if minute >= session_len - 1:            return "post_workout"
    if activity == "weightlifting":
        if effort_pct >= 0.80:               return "final_rep"
        if effort_pct >= 0.55:               return "working_set"
        return "rest_set"
    if activity == "hiit":
        if effort_pct >= 0.90 or hr_zone >= 5: return "redline"
        if effort_pct >= 0.55:               return "working_set"
        return "active_recovery"
    # running / cycling
    if effort_pct >= 0.88 or hr_zone >= 5:   return "peak_effort"
    if effort_pct >= 0.74:                   return "breakthrough_wall"
    if effort_pct >= 0.45:                   return "steady_state"
    if effort_pct >= 0.25:                   return "early_push"
    return "recovery_window"


def delivery_for(moment: str):
    """(format, mode) — mode is 'interrupt' (haptic + verse) or 'ambient' (glow only)."""
    return DELIVERY.get(moment, ("display", "ambient"))


# internal ref book-codes → YouVersion USFM codes (only where they differ).
# Passage ids use USFM codes, e.g. Philippians is PHP (not PHI).
_YV_BOOK_FIX = {"PHI": "PHP"}


def _to_yv_ref(ref):
    """Map an internal ref (e.g. 'PHI.4.13') to a YouVersion USFM passage id ('PHP.4.13')."""
    dot = ref.find(".")
    book = ref[:dot] if dot > 0 else ref
    return _YV_BOOK_FIX[book] + ref[dot:] if book in _YV_BOOK_FIX else ref


# English public-domain default: bibleId 3034 = Berean Standard Bible (BSB).
YV_DEFAULT_BIBLE_ID = 3034


def get_verse(ref, translation="WEB", language="en", *, live=False,
              api_key="", api_base="https://api.youversion.com/v1",
              bible_id=YV_DEFAULT_BIBLE_ID, requests=None):
    """DEMO: public-domain (WEB) mirror. LIVE: YouVersion Platform API.

    Live path mirrors proxy/src/worker.js:
      GET {api_base}/bibles/{bible_id}/passages/{passage_id}
      header  X-YVP-App-Key: <key>   (NOT Authorization: Bearer)
      Accept: application/json  →  verse text is in the response `.content` field.
    English default bible_id = 3034 (Berean Standard Bible, public domain).
    """
    if not live or requests is None:
        return {"reference": ref, "name": VERSE_NAME.get(ref, ref),
                "text": DEMO_VERSES_PD.get(ref, ""), "translation": "WEB (public domain)"}
    passage_id = _to_yv_ref(ref)
    r = requests.get(f"{api_base}/bibles/{bible_id}/passages/{passage_id}",
                     headers={"X-YVP-App-Key": api_key, "Accept": "application/json"},
                     timeout=10)
    r.raise_for_status()
    d = r.json()
    return {"reference": ref, "name": VERSE_NAME.get(ref, ref),
            "text": d.get("content", ""), "translation": translation}


# Gloo real endpoints (see proxy/src/worker.js — the reference implementation).
GLOO_TOKEN_URL = "https://platform.ai.gloo.com/oauth2/token"
GLOO_CHAT_URL = "https://platform.ai.gloo.com/ai/v2/chat/completions"


def _gloo_token(api_key, requests):
    """Exchange Gloo client-credentials for a short-lived bearer token.

    `api_key` may be "CLIENT_ID:CLIENT_SECRET" (real OAuth2 flow) or an already-minted
    bearer token (returned as-is)."""
    if ":" not in api_key:
        return api_key  # already a bearer token
    client_id, client_secret = api_key.split(":", 1)
    r = requests.post(GLOO_TOKEN_URL,
                      data={"grant_type": "client_credentials", "scope": "api/access"},
                      auth=(client_id, client_secret), timeout=15)
    r.raise_for_status()
    return r.json()["access_token"]


def personalize(moment, verse, name="Maya", *, live=False,
                api_key="", requests=None):
    """DEMO: pastoral template. LIVE: Gloo AI (faith-tuned, OpenAI-compatible).

    Live path mirrors proxy/src/worker.js: OAuth2 client-credentials token exchange at
    platform.ai.gloo.com/oauth2/token, then POST platform.ai.gloo.com/ai/v2/chat/completions
    with Authorization: Bearer <token>, body {messages:[...]} (model optional/auto-routed).
    NOTE: Gloo is currently gated by a billing decline, so the deployed proxy runs an
    honest labeled simulation (source:"gloo-sim") — see docs/GLOO-STATUS.md."""
    if not live or requests is None:
        return DEMO_NOTES.get(moment, "Keep going — He's with you.")
    token = _gloo_token(api_key, requests)
    prompt = (f"In one short, warm sentence a runner can read at a glance during "
              f"'{moment.replace('_',' ')}', encourage {name} with this verse: "
              f"\"{verse['text']}\" ({verse['reference']}). Pastoral, no preamble.")
    r = requests.post(GLOO_CHAT_URL,
                      headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"},
                      json={"messages":[{"role":"user","content":prompt}], "max_tokens":40}, timeout=15)
    r.raise_for_status()
    return r.json()["choices"][0]["message"]["content"].strip()


def deliver(snap: Snapshot, name="Maya", *, live=False, yv_key="", gloo_key="", requests=None):
    """The full loop, once per snapshot."""
    m = classify_moment(snap.hr_zone, snap.effort_pct, snap.activity_type, snap.session_minute)
    ref, theme = MOMENT_VERSE.get(m, ("PHI.4.13", "strength"))
    verse = get_verse(ref, snap.translation, snap.language, live=live, api_key=yv_key, requests=requests)
    note  = personalize(m, verse, name, live=live, api_key=gloo_key, requests=requests)
    fmt, mode = delivery_for(m)
    return {"moment": m, "theme": theme, "color": THEME_COLOR.get(theme, "#38e1c0,#14b8a6"),
            "reference": verse["reference"], "name": verse["name"], "verse": verse["text"],
            "note": note, "format": fmt, "mode": mode,
            "hr": snap.heart_rate, "minute": snap.session_minute}

print("engine loaded ·", len(MOMENT_VERSE), "moments ·", len(DEMO_VERSES_PD), "public-domain verses")

## The data — every kind of effort

Kaggle attaches the competition data at `/kaggle/input`; we fall back to the local
file so the notebook also runs in this repo.

In [ ]:
# ═══ 3 · Load the biometric sessions ══════════════════════════════════
import pandas as pd, numpy as np
def _find(name):
    for p in [f"/kaggle/input/scripture-in-new-frontiers/{name}",
              f"/kaggle/input/scripture-new-frontiers/{name}", name]:
        if os.path.exists(p):
            return p
    return name
bio  = pd.read_csv(_find("biometric movements.csv"))
vmap = pd.read_csv(_find("verse movement mapping.csv"))
print(f"{len(bio)} snapshots · {bio.session_id.nunique()} sessions · "
      f"{bio.activity_type.nunique()} activities · {bio.moment_type.nunique()} moment types")
print("\nsessions:")
print(bio.groupby("session_id").agg(activity=("activity_type","first"),
                                     minutes=("session_minute","max"),
                                     snapshots=("moment_type","size")))
bio.head()

## Sense — prove the classifier generalizes across *people*

A wearable must name the moment on-device and never fire at the wrong one. So we
don't grade on random rows — we grade on **held-out people**. With only 5 sessions,
random splits leak a runner's own rows into both train and test; instead we use
**`GroupKFold` on `session_id`** so every prediction is made for a session the model
never trained on. That is the honest question: *does this transfer to a body it has
never seen?*

In [ ]:
# ═══ 4 · RandomForest, evaluated held-out BY SESSION (GroupKFold) ═════
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

FEATURES = ["heart_rate", "hr_zone", "effort_pct", "recovery_score", "stress_index"]
X = bio[FEATURES]
y = bio["moment_type"]
groups = bio["session_id"]

clf = RandomForestClassifier(n_estimators=300, max_depth=8,
                             class_weight="balanced", random_state=7)

# Every fold holds out whole sessions → predictions are for unseen people.
gkf = GroupKFold(n_splits=bio.session_id.nunique())
y_pred = cross_val_predict(clf, X, y, cv=gkf, groups=groups)
heldout_acc = accuracy_score(y, y_pred)

clf.fit(X, y)   # final model trained on everything, for inference below
print(f"Held-out-by-session accuracy: {heldout_acc:.2f}  "
      f"({(y_pred==y).sum()}/{len(y)} moments, across {bio.session_id.nunique()} unseen sessions)")
print("\nWhat the body tells the model (feature importance):")
for f, imp in sorted(zip(FEATURES, clf.feature_importances_), key=lambda t:-t[1]):
    print(f"  {f:15s} {imp:.2f}")

### Per-moment precision & recall

Accuracy hides which moments are hard. The classification report (on the held-out
predictions) shows precision/recall per moment — the interrupt moments (`peak_effort`,
`breakthrough_wall`, `final_rep`) are the ones that must not misfire.

In [ ]:
# ═══ 5 · Per-moment precision / recall on the held-out predictions ════
report = classification_report(y, y_pred, zero_division=0, digits=2)
print(report)

rep = classification_report(y, y_pred, zero_division=0, output_dict=True)
prf = (pd.DataFrame(rep).T
         .loc[[m for m in sorted(y.unique()) if m in rep]]
         [["precision","recall","f1-score","support"]]
         .sort_values("recall", ascending=False))
prf

### Where it confuses one moment for another

A confusion matrix on the held-out predictions. Rows are the true moment, columns the
predicted one; a strong diagonal means the model reads the body correctly for people it
has never seen. Off-diagonal cells are almost always *adjacent* intensities (e.g.
`steady_state` ↔ `early_push`) — a graceful failure mode for a Scripture wearable.

In [ ]:
# ═══ 6 · Confusion matrix — dark, brand-teal sequential ═══════════════
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# Brand palette
BG, INK, INK2, MUTED = "#0a0c10", "#eef1f5", "#c3c2b7", "#7f8794"
GRID, ACCENT = "#2a2f38", "#38e1c0"
# Sequential single-hue teal ramp: near-zero recedes into the dark surface.
teal_seq = LinearSegmentedColormap.from_list(
    "selah_teal", ["#11161c", "#123b3a", "#1c7d72", "#38e1c0"])

labels = sorted(y.unique())
cm = confusion_matrix(y, y_pred, labels=labels)

fig, ax = plt.subplots(figsize=(9.2, 8), dpi=120)
fig.patch.set_facecolor(BG); ax.set_facecolor(BG)
vmax = cm.max()
im = ax.imshow(cm, cmap=teal_seq, vmin=0, vmax=vmax, aspect="equal")

# 2px surface gap between cells (grid on minor ticks in the surface color)
ax.set_xticks(np.arange(-.5, len(labels), 1), minor=True)
ax.set_yticks(np.arange(-.5, len(labels), 1), minor=True)
ax.grid(which="minor", color=BG, linewidth=2)
ax.tick_params(which="minor", length=0)

# Count labels — white on dark cells, ink on bright cells (by luminance).
for i in range(len(labels)):
    for j in range(len(labels)):
        v = cm[i, j]
        if v == 0:
            continue
        tone = INK if (v / vmax) < 0.55 else BG
        ax.text(j, i, str(v), ha="center", va="center", fontsize=9,
                color=tone, fontweight="bold" if i == j else "normal")

ax.set_xticks(range(len(labels))); ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=8, color=MUTED)
ax.set_yticklabels(labels, fontsize=8, color=MUTED)
ax.set_xlabel("predicted moment", color=INK2, fontsize=10)
ax.set_ylabel("true moment", color=INK2, fontsize=10)
ax.set_title("Selah · moment confusion — held out by session",
             color=INK, loc="left", fontsize=13, pad=12)
for s in ax.spines.values():
    s.set_color(GRID)
cbar = fig.colorbar(im, ax=ax, fraction=0.045, pad=0.03)
cbar.set_label("snapshots", color=MUTED, fontsize=9)
cbar.ax.yaxis.set_tick_params(color=GRID); cbar.outline.set_edgecolor(GRID)
plt.setp(cbar.ax.get_yticklabels(), color=MUTED)
plt.tight_layout(); plt.show()

## Discern — the verse for the moment, from YouVersion

Each moment maps to Scripture built for it (`MOMENT_VERSE` in the engine, extending
the hackathon's verse mapping). Verse text comes **live from the YouVersion Platform
API** — 2,000+ languages, and critically the reader's *own* licensed translation
(NIV, ESV, NLT…).

> **IP-safe by design.** Because translations like NIV/ESV are copyrighted, this
> notebook's demo mode never ships them. `get_verse(...)` returns **public-domain**
> World English Bible text offline; the licensed translation is only ever fetched
> live, per-reader, from YouVersion — exactly how a shipping app must handle it.

In [ ]:
# ═══ 7 · Moment → verse (engine-driven, public-domain in demo) ════════
demo_moments = ["breakthrough_wall", "peak_effort", "recovery_window", "post_workout"]
for m in demo_moments:
    ref, theme = MOMENT_VERSE[m]
    v = get_verse(ref, live=not DEMO_MODE, api_key=YOUVERSION_API_KEY, requests=requests)
    print(f"{m:<18} → {v['name']:<16} [{theme}]  ({v['translation']})")
    print(f"    “{v['text']}”\n")

## Innovative API use — Selah reads *more* of YouVersion than a single verse

A verse lookup is the floor. Selah is designed to meet the reader inside their existing
YouVersion life, so it pulls from three richer surfaces of the Platform API — each with
a real endpoint shape and a demo fallback:

1. **Verse of the Day** — anchor the warm-up in what the reader is *already* reading today.
2. **Active reading plan** — if they're mid-plan (e.g. a 21-day endurance plan), the
   peak-effort verse can come from *their* plan, not a generic map.
3. **Highlights** — verses the reader has personally highlighted are prioritized, so
   Scripture that already moved them returns at the moment their body needs it most.

In [ ]:
# ═══ 8 · Broader YouVersion Platform surfaces (real shapes + demo) ════
def yv_get(path, params=None):
    """GET a YouVersion Platform endpoint (live), or return None in demo mode."""
    if DEMO_MODE or requests is None:
        return None
    r = requests.get(f"{YOUVERSION_API_BASE}{path}",
                     headers={"X-YVP-App-Key": YOUVERSION_API_KEY,   # app-key header, NOT Bearer
                              "Accept": "application/json"},
                     params=params or {}, timeout=10)
    r.raise_for_status()
    return r.json()

# The three surfaces below are roadmap features — their exact live paths/shapes are not
# fully published yet (see docs/SETUP-APIS.md), so each carries an illustrative shape plus
# a public-domain demo fallback. The verse/passage endpoint (get_verse) is the confirmed one.
def verse_of_the_day(translation="WEB"):
    # LIVE (roadmap)  GET /verse_of_the_day  →  {"reference": "...", "content": "..."}
    data = yv_get("/verse_of_the_day", {"translation": translation})
    if data:
        return {"reference": data["reference"], "text": data.get("content", ""), "source": "YouVersion VOTD"}
    ref = "PSA.118.24"                       # demo anchor
    return {**get_verse(ref), "source": "demo VOTD (public-domain)"}

def verse_from_reading_plan(user_id, moment):
    # LIVE (roadmap)  reading-plan sub-resource  →  {"plan_id","day","references":[...]}
    plan = yv_get(f"/users/{user_id}/reading_plans/active")
    if plan and plan.get("references"):
        ref = plan["references"][0]          # pick a plan verse aligned to today
        return {**get_verse(ref), "source": f"reading plan {plan['plan_id']} · day {plan['day']}"}
    ref, _ = MOMENT_VERSE[moment]            # demo fallback → moment map
    return {**get_verse(ref), "source": "demo (no active plan)"}

def verse_from_highlights(user_id, theme):
    # LIVE (roadmap)  user highlights (needs user auth)  →  {"items":[{"reference","color"},...]}
    hl = yv_get(f"/users/{user_id}/highlights", {"theme": theme})
    if hl and hl.get("items"):
        ref = hl["items"][0]["reference"]    # a verse THEY highlighted, on-theme
        return {**get_verse(ref), "source": "your highlight"}
    return {**get_verse("ISA.40.31"), "source": "demo highlight (public-domain)"}

votd = verse_of_the_day()
plan = verse_from_reading_plan("maya", "peak_effort")
high = verse_from_highlights("maya", "endurance")
for label, v in [("Verse of the Day", votd), ("From reading plan", plan), ("From highlights", high)]:
    print(f"{label:<18} {v['name']:<16} — {v['source']}")
    print(f"    “{v['text']}”\n")

## Personalize — one pastoral line from Gloo AI Studio

The engine's `personalize(...)` calls **Gloo AI Studio's chat/inference endpoint** with a
tightly-scoped prompt: one warm sentence a runner can read at a glance, grounded in the
retrieved verse, faith-tuned, no preamble. Below is the exact request shape and the prompt
Gloo receives — plus the demo template it falls back to offline.

In [ ]:
# ═══ 9 · Gloo AI Studio — the real request shape + prompt ═════════════
# Live flow (see proxy/src/worker.js): OAuth2 client-credentials token exchange at
# GLOO_TOKEN_URL, then POST GLOO_AI_API_BASE/chat/completions with Bearer <token>.
def gloo_prompt(moment, verse, name="Maya"):
    return (f"In one short, warm sentence a runner can read at a glance during "
            f"'{moment.replace('_',' ')}', encourage {name} with this verse: "
            f"\"{verse['text']}\" ({verse['reference']}). Pastoral, no preamble.")

_m = "breakthrough_wall"
_ref, _ = MOMENT_VERSE[_m]
_verse = get_verse(_ref, live=not DEMO_MODE, api_key=YOUVERSION_API_KEY, requests=requests)

print("POST", f"{GLOO_AI_API_BASE}/chat/completions")
print(json.dumps({
    # `model` is optional — Gloo auto-routes when omitted.
    "messages": [{"role": "user", "content": gloo_prompt(_m, _verse)}],
    "max_tokens": 40,
}, indent=2))

line = personalize(_m, _verse, "Maya", live=not DEMO_MODE,
                   api_key=GLOO_AI_API_KEY, requests=requests)
print("\n→ Gloo returns:", repr(line))

## Smoke test — the integration is wired, not faked

This cell attempts a **real** call to each API when its key is present, prints the
result, and **fails gracefully** with a clear message when a key is missing or the
network is unavailable. It is proof the wiring is real: give it keys and it lights up.

In [ ]:
# ═══ 10 · Live smoke test (safe: no-op when keys absent) ══════════════
def smoke_youversion():
    if not YOUVERSION_API_KEY or requests is None:
        return "SKIP · no YouVersion key set (demo mode serves public-domain WEB)"
    try:
        # English default = bibleId 3034 (Berean Standard Bible, public domain).
        v = get_verse("PHI.4.13", live=True, api_key=YOUVERSION_API_KEY,
                      bible_id=YV_BIBLE_ID, requests=requests)
        return f"OK · YouVersion returned {v['reference']} [BSB]: “{v['text'][:60]}…”"
    except Exception as e:
        return f"FAIL · YouVersion call errored: {type(e).__name__}: {e}"

def smoke_gloo():
    if not GLOO_AI_API_KEY or requests is None:
        return "SKIP · no Gloo key set (demo mode serves pastoral template)"
    try:
        v = get_verse("PHI.4.13")
        line = personalize("breakthrough_wall", v, "Maya", live=True,
                           api_key=GLOO_AI_API_KEY, requests=requests)
        return f"OK · Gloo returned: “{line}”"
    except Exception as e:
        return f"FAIL · Gloo call errored: {type(e).__name__}: {e}"

print("YouVersion:", smoke_youversion())
print("Gloo AI   :", smoke_gloo())
print("\nDemo mode runs everything above with zero keys — the calls are wired, "
      "not faked; add keys to light them up.")

## Deliver — Scripture meeting every kind of effort

Selah is not a running app. The same engine serves a runner grinding through the wall
and a lifter on a final rep. We run the full `deliver(...)` loop across **two very
different sessions** — S001 (running) and S003 (weightlifting) — building an engine
`Snapshot` from each biometric row and letting the engine sense, discern, and deliver.

Note the two **delivery modes**: `interrupt` (a haptic pulse + a verse to read *now*, at
peak effort) versus `ambient` (the watch just breathes a color, never interrupting
recovery).

In [ ]:
# ═══ 11 · Run the pipeline across running AND weightlifting ═══════════
def snap_from_row(r):
    return Snapshot(heart_rate=int(r.heart_rate), hr_zone=int(r.hr_zone),
                    effort_pct=float(r.effort_pct), recovery_score=int(r.recovery_score),
                    stress_index=float(r.stress_index), activity_type=str(r.activity_type),
                    session_minute=int(r.session_minute))

def run_session(sid, name):
    s = bio[bio.session_id == sid].sort_values("session_minute")
    act = s.activity_type.iloc[0]
    print(f"══ {sid} · {act.upper()} · {name} " + "═"*(46-len(act)-len(name)))
    rows = []
    for _, r in s.iterrows():
        d = deliver(snap_from_row(r), name=name, live=not DEMO_MODE,
                    yv_key=YOUVERSION_API_KEY, gloo_key=GLOO_AI_API_KEY, requests=requests)
        d["minute"] = int(r.session_minute); d["hr"] = int(r.heart_rate)
        rows.append(d)
        tag = "▶ INTERRUPT" if d["mode"] == "interrupt" else "· ambient  "
        print(f"[{d['minute']:>2}m {d['hr']:>3}bpm] {tag} {d['moment']:<17} "
              f"{d['name']:<16} — {d['format']}")
        if d["mode"] == "interrupt":
            print(f"            “{d['verse']}”")
            print(f"            › {d['note']}")
    print()
    return rows

run_rows  = run_session("S001", "Maya")
lift_rows = run_session("S003", "David")

### See both — Scripture landing on two different bodies

Small multiples, one panel per session: the effort curve, with each Scripture delivery
marked at the moment it fired. **Filled markers = interrupt** (haptic + verse); **hollow
markers = ambient** (glow only). The interrupt deliveries cluster exactly where the body
is under the most load — the wall, the peak, the final rep.

In [ ]:
# ═══ 12 · Visualize both sessions ════════════════════════════════════
INTERRUPT, AMBIENT = "#ff8a3d", "#38e1c0"   # 2 categories: mode
panels = [("S001", "running · Maya", run_rows), ("S003", "weightlifting · David", lift_rows)]

fig, axes = plt.subplots(2, 1, figsize=(11, 7.4), dpi=120)
fig.patch.set_facecolor(BG)
for ax, (sid, title, rows) in zip(axes, panels):
    s = bio[bio.session_id == sid].sort_values("session_minute")
    mins = s.session_minute.values; hr = s.heart_rate.values
    ax.set_facecolor(BG)
    ax.plot(mins, hr, color=ACCENT, lw=2, zorder=2, solid_capstyle="round")
    ax.fill_between(mins, hr, hr.min()-6, color=ACCENT, alpha=0.10, zorder=1)
    for d in rows:
        interrupt = d["mode"] == "interrupt"
        ax.scatter(d["minute"], d["hr"], s=95, zorder=4,
                   facecolors=INTERRUPT if interrupt else "none",
                   edgecolors=INTERRUPT if interrupt else AMBIENT,
                   linewidths=2)
        if interrupt:   # label sparingly — only the moments that interrupt
            ax.annotate(d["reference"].split(".")[0], (d["minute"], d["hr"]),
                        textcoords="offset points", xytext=(0, 11), ha="center",
                        fontsize=8, color=INK2, fontweight="bold")
    ax.set_title(f"Selah · {title}", color=INK, loc="left", fontsize=12, pad=8)
    ax.set_ylabel("heart rate (bpm)", color=MUTED, fontsize=9)
    ax.tick_params(colors=MUTED, labelsize=8)
    for k in ("top", "right"): ax.spines[k].set_visible(False)
    for k in ("left", "bottom"): ax.spines[k].set_color(GRID)
axes[-1].set_xlabel("session minute", color=MUTED, fontsize=9)

# Legend (identity is never color-alone: fill vs hollow + labels)
from matplotlib.lines import Line2D
legend = [Line2D([0],[0], marker="o", linestyle="none", markersize=9,
                 markerfacecolor=INTERRUPT, markeredgecolor=INTERRUPT, label="interrupt · haptic + verse"),
          Line2D([0],[0], marker="o", linestyle="none", markersize=9,
                 markerfacecolor="none", markeredgecolor=AMBIENT, markeredgewidth=2, label="ambient · glow only")]
axes[0].legend(handles=legend, loc="upper left", frameon=False,
               labelcolor=INK2, fontsize=8.5)
fig.suptitle("Scripture meeting every kind of effort", color=INK, x=0.012, ha="left",
             fontsize=14, y=0.99)
plt.tight_layout(rect=[0, 0, 1, 0.97]); plt.show()

---
## Selah

**Sense** the moment on-device — proven to generalize to bodies it has never seen.
**Discern** the verse from YouVersion — the reader's own translation, their plan, their
highlights, their verse of the day. **Deliver** one pastoral line from Gloo AI, in the
moment's native format — a haptic verse at the wall, an ambient glow in the quiet after.

Everything above ran in **demo mode with zero keys**, on public-domain Scripture, wired
to the real endpoints. Add your YouVersion and Gloo keys, set `DEMO_MODE = False`, and
the same pipeline lights up live — Scripture that meets you at the wall, the peak, and
the quiet after.

*Built with the YouVersion Platform API and the Gloo AI Studio API. Live demo, writeup,
and video in the submission.*